# **5.2 Parameter Management**
Once we have selected the architecture and set the hyperparameters, we move on to the training phase. At this stage, our objective is to find the model parameter values that minimise the loss function. Following training, we will need to use these parameters to make future predictions. Furthermore, we may sometimes wish to extract the parameters so that they can be reused in other contexts, save the model so that it can be run in other software, or examine it to gain a scientific understanding.<br><br>
In this section, we will cover the following:
  * Access parameters, used for debugging, diagnostics and visualization;
  * Parameter initialization;
  * Share parameters between different model components.

Let us first consider a multi-layer perceptron with a single hidden layer.

In [1]:
import torch
from torch import nn

net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
X = torch.randn(size=(2, 4))
net(X)

tensor([[-0.4447],
        [ 0.1763]], grad_fn=<AddmmBackward0>)

## **5.2.1 Parameter Access**
We access parameters from existing models. When defining a model using the `Sequential` class, we can access any layer of the model via its index. It is as if the model were a list, with the parameters for each layer contained within its properties. As shown below, we can examine the parameters of the second fully connected layer.

In [2]:
print(net[2].state_dict())

OrderedDict({'weight': tensor([[ 0.2794,  0.0038, -0.1548,  0.3532,  0.2503,  0.1967, -0.3531, -0.1785]]), 'bias': tensor([0.0605])})


The output results reveal some important points: firstly, this fully connected layer contains two parameters, namely the layer’s weights and biases. Both are stored as single-precision floating-point numbers (float32). **Note that the parameter names allow each parameter to be uniquely identified, even in networks comprising hundreds of layers.**

### **Target parameters**
Please note that each parameter is represented as an instance of the parameter class. To perform any operation on a parameter, we first need to access the underlying numerical value. There are several ways to do this. Some are relatively simple, whilst others are more general-purpose. The code below extracts the bias from the second fully connected layer (i.e. the third layer of the neural network); the result is an instance of the parameter class, from which we can then access the parameter’s value.

In [3]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.0605], requires_grad=True)
tensor([0.0605])


Parameters are composite objects comprising values, gradients and additional information. This is why we need explicit parameter values. In addition to the values, we can also access the gradients for each parameter. **In the network shown above, as we have not yet performed backpropagation, the gradients of the parameters are in their initial state**.

In [4]:
net[2].weight.grad == None

True

### **Access all parameters in a single call**
When we need to perform an operation on all parameters, accessing them one by one can be rather cumbersome. This can become particularly complicated when dealing with more complex blocks (such as nested blocks), as we need to traverse the entire tree recursively to extract the parameters of each sub-block. Below, we shall compare, through a demonstration, accessing the parameters of the first fully connected layer with accessing those of all layers.

In [5]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


This provides us with another way of accessing network parameters, as shown below.

In [6]:
net.state_dict()['2.bias'].data

tensor([0.0605])

### **Collecting parameters from nested blocks**
How does the parameter naming convention work when nesting multiple blocks within one another? We first define a function that generates blocks (which could be called a ‘block factory’), and then combine these blocks into a larger block.

In [7]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module('block ' + str(i), block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.4245],
        [-0.4247]], grad_fn=<AddmmBackward0>)

In [8]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


**As the layers are nested, we can also access them in the same way as we would index through a nested list**. Below, we access the item at the first level of the second sub-block within the first main block.

In [9]:
rgnet[0][1][0].bias.data

tensor([ 0.3532,  0.2039, -0.4094,  0.4226, -0.4256,  0.3222, -0.3539,  0.0822])

## **5.2.2 Parameter initialization**
Now that we know how to access the parameters, let us look at how to initialise them correctly. We discussed the necessity of proper initialisation in [Section 4.8](../../chapter_multi_layer_perceptron/content/numerical_stability_and_model_initialization.ipynb#48-numerical-stability-and-model-initialization). **Deep learning frameworks provide default random initialisation, whilst also allowing us to create custom initialisation methods to satisfy our requirements for initialising weights according to other rules**.<br><br>
**By default, PyTorch initialises weights and bias matrices uniformly across a range calculated based on the input and output dimensions**. PyTorch’s `nn.init` module provides a variety of predefined initialisation methods.

### **Built-in initialisation**
Let us first call the built-in initialiser. The code below initialises all weight parameters as Gaussian random variables with a standard deviation of 0.01, and sets the bias parameters to 0.

In [10]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0102, -0.0086, -0.0130, -0.0059]), tensor(0.))

We can also initialise all parameters to a given constant, such as 1.

In [11]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

We can also apply different initialisation methods to specific blocks. For example, below we initialise the first neural network layer using the Xavier initialisation method, and then initialise the third neural network layer with the constant value 42.

In [12]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.1866,  0.0350, -0.6826, -0.0512])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### **Custom initialisation**
Sometimes, deep learning frameworks do not provide the initialisation methods we require. In the example below, we use the following distribution to define an initialisation method for arbitrary weight parameters $w$:
##### **(5.2.1)**
$$
w \sim
\begin{cases}
  U(5,10),      & possibility \;\; \frac{1}{4}, \\
  0,        & possibility \;\; \frac{1}{2}, \\
  U(-10,-5),       & possibility \;\; \frac{1}{4}.
\end{cases}
$$
Similarly, we have implemented a `my_init` function to be applied to `net`.

In [14]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 7.2106,  6.4442, -6.1304,  0.0000],
        [ 7.1929,  0.0000, -0.0000, -0.0000]], grad_fn=<SliceBackward0>)

Please note that we can always set the parameters directly.

In [15]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000,  7.4442, -5.1304,  1.0000])

## **5.2.3 Parameter binding**
Sometimes we wish to share parameters across multiple layers: we can define a dense layer and then use its parameters to set the parameters of another layer.

In [16]:
# We need to give the shared layer a name so that we can refer to its parameters
shared = nn.Linear(8, 8)
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    shared, nn.ReLU(),
    shared, nn.ReLU(),
    nn.Linear(8, 1)
)

net(X)
# check if the parameters are the same
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# Ensure that they are in fact the same object, and not merely objects with the same values
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


This example illustrates that the parameters of the third and fifth layers of the neural network are bound. Not only are their values equal, but they are also represented by the same tensor. Consequently, if we change one of these parameters, the other will also change. This raises a question: what happens to the gradients when parameters are bound? **The answer is that, as the model parameters contain gradients, during backpropagation the gradients of the second hidden layer (i.e. the third layer of the neural network) and the third hidden layer
(i.e. the fifth layer of the neural network) are added together**.

## **Summary**
  * There are several ways to access, initialise and bind model parameters.
  * We can use a custom initialisation method.